# Game-Theoretic Threat Estimator — v3

**v3 = v2 core + Bayesian Initial Reputation + Adaptive T_S Thresholds**

This notebook extends v2 with two capabilities drawn from the 8 April plan ("Discussion on initial reputation calculation using Bayesian modelling & intuition behind threat score threshold values"):

1. **Bayesian Initial Reputation (§6 – §7).** Solves the *cold-start* problem: a brand-new drone's reputation $R$ is derived from $P(\text{Benign} \mid \text{Zone}, \text{FileType})$ via a small Bayesian Belief Network, replacing the static default of $0.8$ used in v2.
2. **Adaptive $T_S$ Thresholds (§8).** Replaces the fixed $0.4 / 0.7$ inspection cut-offs with a soft-update rule driven by live FPR / FNR feedback from the Response & Quarantine Manager.
3. **Integrated v3 Pipeline (§9).** Wires Bayesian $R$ into Step 2 and routes Step 4's $T_S$ through the `AdaptiveThresholdManager` before selecting the inspection level.

Everything in Steps 1 – 5 (the game-theoretic Stackelberg solver) is preserved verbatim from v2; the new sections wrap it rather than replace it.

---


# Sample Drone Payloads

Sample A — Video + Image (normal)

In [1]:
# %%JSON
# {
#   "drone_id": "DRN-001",
#   "timestamp": "2025-10-13T03:00:12Z",
#   "mission_id": "MSN-142",
#   "mission_zone": "zone-a",
#   "geo": { "lat": 12.971598, "lon": 77.594566, "alt": 120 },
#   "payloads": [
#     {
#       "type": "video",
#       "filename": "drn001_fpv_001.mp4",
#       "mime": "video/mp4",
#       "size_bytes": 4500000,
#       "encryption": false,
#       "container": false,
#       "checksum": "a1b2c3d4..."
#     },
#     {
#       "type": "image",
#       "filename": "drn001_cam_001.jpg",
#       "mime": "image/jpeg",
#       "size_bytes": 320000,
#       "encryption": false,
#       "container": false,
#       "checksum": "e5f6g7h8..."
#     }
#   ],
#   "telemetry": { "speed": 12.5, "heading": 145.2, "battery": 78.4, "signal_strength": 82.1 },
#   "signature": null,
#   "firmware_version": "v1.2.0",
#   "operator_id": "OP-12",
#   "additional_metadata": { "camera_model": "CAM-X1000", "frame_rate": 30 }
# }


Sample B — Encrypted nested archive (suspicious-looking)

In [2]:
# %%JSON{
#   "drone_id": "DRN-002",
#   "timestamp": "2025-10-13T03:05:45Z",
#   "mission_id": "MSN-143",
#   "mission_zone": "zone-c",
#   "geo": { "lat": 13.035542, "lon": 77.597100, "alt": 85 },
#   "payloads": [
#     {
#       "type": "archive",
#       "filename": "payload_bundle.zip",
#       "mime": "application/zip",
#       "size_bytes": 4200000,
#       "encryption": true,
#       "container": true,
#       "checksum": "9f8e7d6c..."
#     },
#     {
#       "type": "text",
#       "filename": "notes.txt",
#       "mime": "text/plain",
#       "size_bytes": 2048,
#       "encryption": false,
#       "container": false,
#       "checksum": "1234abcd..."
#     }
#   ],
#   "telemetry": { "speed": 0.0, "heading": 0.0, "battery": 56.1, "signal_strength": 65.3 },
#   "signature": "ed25519:abcdef012345...",
#   "firmware_version": "v1.1.9",
#   "operator_id": "OP-23",
#   "additional_metadata": { "mission_priority": "high", "notes": "compressed mission dataset" }
# }


Sample C — Telemetry-only / small text (low-risk)

In [3]:
# %%JSON
# {
#   "drone_id": "DRN-003",
#   "timestamp": "2025-10-13T03:10:03Z",
#   "mission_id": "MSN-144",
#   "mission_zone": "zone-b",
#   "geo": { "lat": 12.967800, "lon": 77.601200, "alt": 35 },
#   "payloads": [
#     {
#       "type": "telemetry",
#       "filename": "telemetry_snapshot.json",
#       "mime": "application/json",
#       "size_bytes": 1500,
#       "encryption": false,
#       "container": false,
#       "checksum": "fedcba987..."
#     }
#   ],
#   "telemetry": { "speed": 6.2, "heading": 220.0, "battery": 92.3, "signal_strength": 90.4 },
#   "signature": null,
#   "firmware_version": "v1.2.3",
#   "operator_id": "OP-33",
#   "additional_metadata": { "note": "routine patrol", "weather": "clear" }
# }


Sample D — Mixed with large video + camera metadata (mission-critical)

In [4]:
# %%JSON
# {
#   "drone_id": "DRN-004",
#   "timestamp": "2025-10-13T03:15:22Z",
#   "mission_id": "MSN-145",
#   "mission_zone": "zone-a",
#   "geo": { "lat": 12.975000, "lon": 77.590000, "alt": 200 },
#   "payloads": [
#     {
#       "type": "video",
#       "filename": "survey_coverage_long.mp4",
#       "mime": "video/mp4",
#       "size_bytes": 12500000,
#       "encryption": false,
#       "container": false,
#       "checksum": "aaaabbbbcccc..."
#     },
#     {
#       "type": "image",
#       "filename": "survey_frame_2345.jpg",
#       "mime": "image/jpeg",
#       "size_bytes": 550000,
#       "encryption": false,
#       "container": false,
#       "checksum": "ddddeeeeffff..."
#     }
#   ],
#   "telemetry": { "speed": 8.1, "heading": 98.7, "battery": 64.0, "signal_strength": 75.0 },
#   "signature": "ed25519:98765fedcba...",
#   "firmware_version": "v2.0.0",
#   "operator_id": "OP-05",
#   "additional_metadata": { "camera_model": "CAM-PRO-4k", "mission_sensitivity": "critical" }
# }


## 2. Sample outputs from the Ingestion Interceptor

Ingest Output for Sample A (DRN-001)

In [5]:
# %%JSON
# {
#   "ingest_metadata": {
#     "ingest_id": "ingest_9f1a2b3c4d",
#     "drone_id": "DRN-001",
#     "timestamp": "2025-10-13T03:00:12Z",
#     "mission_id": "MSN-142",
#     "mission_zone": "zone-a",
#     "geo": { "lat": 12.971598, "lon": 77.594566, "alt": 120 },
#     "operator_id": "OP-12",
#     "firmware_version": "v1.2.0",
#     "num_files": 2,
#     "insecure_flags": [],
#     "auth_result": "ok",
#     "notes": "normal video+image feed"
#   },
#   "artifact_records": [
#     {
#       "artifact_id": "artifact://a3f8e9b2c1d4",
#       "filename": "drn001_fpv_001.mp4",
#       "type": "video",
#       "mime": "video/mp4",
#       "size_bytes": 4500000,
#       "encryption": false,
#       "container": false,
#       "thumbnail": "thumb://6d7a8b9c0d",
#       "pointer_storage": "s3://forensics/artifacts/a3f8e9b2c1d4"
#     },
#     {
#       "artifact_id": "artifact://b4c5d6e7f8a9",
#       "filename": "drn001_cam_001.jpg",
#       "type": "image",
#       "mime": "image/jpeg",
#       "size_bytes": 320000,
#       "encryption": false,
#       "container": false,
#       "thumbnail": "thumb://1a2b3c4d5e",
#       "pointer_storage": "s3://forensics/artifacts/b4c5d6e7f8a9"
#     }
#   ]
# }


Ingest Output for Sample B (DRN-002 — encrypted nested archive)

In [6]:
# %%JSON
# {
#   "ingest_metadata": {
#     "ingest_id": "ingest_c7d6e5f4a3",
#     "drone_id": "DRN-002",
#     "timestamp": "2025-10-13T03:05:45Z",
#     "mission_id": "MSN-143",
#     "mission_zone": "zone-c",
#     "geo": { "lat": 13.035542, "lon": 77.597100, "alt": 85 },
#     "operator_id": "OP-23",
#     "firmware_version": "v1.1.9",
#     "num_files": 2,
#     "insecure_flags": ["encrypted_payload", "nested_archive"],
#     "auth_result": "unknown",
#     "notes": "encrypted ZIP with nested contents — flag for deferred analysis"
#   },
#   "artifact_records": [
#     {
#       "artifact_id": "artifact://c1d2e3f4a5b6",
#       "filename": "payload_bundle.zip",
#       "type": "archive",
#       "mime": "application/zip",
#       "size_bytes": 4200000,
#       "encryption": true,
#       "container": true,
#       "thumbnail": null,
#       "pointer_storage": "s3://forensics/artifacts/c1d2e3f4a5b6"
#     },
#     {
#       "artifact_id": "artifact://d7e8f9a0b1c2",
#       "filename": "notes.txt",
#       "type": "text",
#       "mime": "text/plain",
#       "size_bytes": 2048,
#       "encryption": false,
#       "container": false,
#       "thumbnail": null,
#       "pointer_storage": "s3://forensics/artifacts/d7e8f9a0b1c2"
#     }
#   ]
# }


Ingest Output for Sample C (DRN-003 — telemetry)

In [7]:
# %%JSON
# {
#   "ingest_metadata": {
#     "ingest_id": "ingest_e8f7g6h5i4",
#     "drone_id": "DRN-003",
#     "timestamp": "2025-10-13T03:10:03Z",
#     "mission_id": "MSN-144",
#     "mission_zone": "zone-b",
#     "geo": { "lat": 12.967800, "lon": 77.601200, "alt": 35 },
#     "operator_id": "OP-33",
#     "firmware_version": "v1.2.3",
#     "num_files": 1,
#     "insecure_flags": [],
#     "auth_result": "ok",
#     "notes": "telemetry-only — low risk"
#   },
#   "artifact_records": [
#     {
#       "artifact_id": "artifact://e9f0g1h2i3j4",
#       "filename": "telemetry_snapshot.json",
#       "type": "telemetry",
#       "mime": "application/json",
#       "size_bytes": 1500,
#       "encryption": false,
#       "container": false,
#       "thumbnail": null,
#       "pointer_storage": "s3://forensics/artifacts/e9f0g1h2i3j4"
#     }
#   ]
# }


Ingest Output for Sample D (DRN-004 — mission-critical large video)

In [8]:
# %%JSON
# {
#   "ingest_metadata": {
#     "ingest_id": "ingest_f1e2d3c4b5",
#     "drone_id": "DRN-004",
#     "timestamp": "2025-10-13T03:15:22Z",
#     "mission_id": "MSN-145",
#     "mission_zone": "zone-a",
#     "geo": { "lat": 12.975000, "lon": 77.590000, "alt": 200 },
#     "operator_id": "OP-05",
#     "firmware_version": "v2.0.0",
#     "num_files": 2,
#     "insecure_flags": [],
#     "auth_result": "ok",
#     "notes": "large survey video — mission sensitivity: critical"
#   },
#   "artifact_records": [
#     {
#       "artifact_id": "artifact://f2e3d4c5b6a7",
#       "filename": "survey_coverage_long.mp4",
#       "type": "video",
#       "mime": "video/mp4",
#       "size_bytes": 12500000,
#       "encryption": false,
#       "container": false,
#       "thumbnail": "thumb://abc123def456",
#       "pointer_storage": "s3://forensics/artifacts/f2e3d4c5b6a7"
#     },
#     {
#       "artifact_id": "artifact://g3h4i5j6k7l8",
#       "filename": "survey_frame_2345.jpg",
#       "type": "image",
#       "mime": "image/jpeg",
#       "size_bytes": 550000,
#       "encryption": false,
#       "container": false,
#       "thumbnail": "thumb://789xyz456",
#       "pointer_storage": "s3://forensics/artifacts/g3h4i5j6k7l8"
#     }
#   ]
# }


NOTE:

ingest_metadata.insecure_flags should be a short set of heuristics the Interceptor detects (e.g., "encrypted_payload", "nested_archive", "unknown_signature", "large_binary"). Use these to increase estimator priors.

artifact_records.pointer_storage shows where the payload is stored for sandbox or forensics (S3/MinIO path). In this notebook we can mock this with artifact:// URIs.

auth_result values: "ok", "unknown", "failed" — feed these as features into the Reputation/Estimator.

## Game Theoretic Threat Estimator

Here’s where the “game” happens:
| Role                     | Player                   | Choices                                       | Examples |
| ------------------------ | ------------------------ | --------------------------------------------- | -------- |
| **Defender (System)**    | Chooses inspection depth | Signature, AI/ML, Sandbox                     |          |
| **Attacker (Adversary)** | Chooses attack strength  | No attack, Simple malware, Obfuscated malware |          |


The system predicts how smart or aggressive the attacker might be and chooses the most cost-effective defense.

### The Logic Behind It

Each defense has a cost (CPU time, latency).

Each attack has a gain or loss depending on whether it succeeds or fails.

The estimator simulates both sides, calculates possible outcomes, and decides the best defense level that minimizes overall risk.

### Game Mathematics in Simple Terms
Each choice pair (Defender’s strategy, Attacker’s move) gives payoffs:
|                | No Attack                       | Attack             |
| -------------- | ------------------------------- | ------------------ |
| Signature Scan | Low cost, low catch rate        | Cheap but risky    |
| AI/ML Scan     | Moderate cost, better detection | Balance            |
| Sandbox        | Expensive, strong catch         | High cost but safe |

The system calculates:

The defender’s utility (U_d) = benefit of detection – cost

The attacker’s utility (U_a) = success of attack – effort/cost

Then finds the Stackelberg equilibrium:
    The defender’s best strategy knowing how the attacker will respond.

### Parameters That Influence the Game

| Parameter  | Meaning                     | Example                                      |
| ---------- | --------------------------- | -------------------------------------------- |
| **R**      | Reputation of drone (trust) | Based on history, 0–1                        |
| **Z**      | Zone risk                   | 0 = safe base, 1 = warzone                   |
| **H**      | Past infection rate         | Higher → more suspicious                     |
| **I_base** | Impact of file type         | Images = low, encrypted ZIPs = high          |
| **C_d**    | Defender cost per strategy  | Sandbox costs more than signature            |
| **DSR**    | Detection success rate      | How effective each layer is                  |
| **T_S**    | Threat score                | Final 0–1 score derived from the equilibrium |


NOTE: More parameters may be added in future.

### Outcome
The estimator produces two outputs:

In [9]:
# %%JSON
# {
#   "Threat_Score": 0.72,
#   "Inspection_Level": "High"
# }


- If T_S < 0.4, → Low (Signature only)

- If 0.4 ≤ T_S < 0.7, → Medium (Signature + ML)

- If T_S ≥ 0.7, → High (Sandbox)

### Intuitive Example

| Drone ID | Reputation | Zone Risk | Past Infection | Threat Score | Decision        |
| -------- | ---------- | --------- | -------------- | ------------ | --------------- |
| DRN-001  | 0.9        | 0.2       | 0.0            | 0.25         | Low (Signature) |
| DRN-007  | 0.6        | 0.7       | 0.3            | 0.65         | Medium (ML)     |
| DRN-013  | 0.3        | 0.8       | 0.6            | 0.85         | High (Sandbox)  |


### Step 1 — Variables, Inputs & Assumptions

These are the variables and parameters required by the estimator for each incoming artifact.

#### Core Inputs from the Ingestion Interceptor

- **drone_id** — *string*. Unique ID for the sending drone.
- **timestamp** — *ISO8601 string*. When the feed was captured.
- **mission_zone** — *string*. Named area or zone id (used to look up zone risk).
- **geo** — *object* `{lat, lon, alt}`. (optional for now)
- **artifact_records** — *list of artifacts* (each with: filename, type, mime, size_bytes, encryption, container, checksum, thumbnail, pointer_storage).

#### Derived / System Inputs (lookup or compute at estimator)

- **R** — *reputation* of the drone/source, numeric in [0, 1].  
    - 1 = fully trusted, 0 = fully untrusted.  
    - Default: 0.8 for new devices.  
    - Source: reputation DB (Redis/Postgres) updated by Feedback Loop.

- **Z** — *zone risk*, numeric in [0, 1].  
    - 0 = safe zone, 1 = highest-risk (e.g., conflict area).  
    - Default: 0.5 (neutral) if unknown.  
    - Source: static policy table or dynamic intelligence.

- **H** — *recent infection frequency* for this drone (or zone), numeric in [0, 1].  
    - Fraction of recent feeds confirmed malicious.  
    - Default: 0.0 if no historic detections.  
    - Example: 0.2 means 20% of last N feeds were malicious.

- **TI_boost** — *threat-intel corroboration strength*, numeric in [0, 1].  
    - 0 if no IOC matches; >0 if indicators match known bad IOCs (higher = stronger corroboration).  
    - Default: 0.0.

- **I_base** — *base impact* of the artifact (0..10).  
    - How bad it would be if this artifact were malicious (mission-criticality + file-type).  
    - Example mapping (starter heuristic):  
        - telemetry / small text → 1–2  
        - images → 3–6  
        - video / large survey → 6–9  
        - encrypted archive with docs → 8–10  
    - Default: compute from artifact_records.

#### Defender / Attacker Model Constants (tunable)

- **DSR(s)** — base detection success rate per defender strategy s (value in (0,1)).  
    - Defaults:  
        - DSR('signature') = 0.70  
        - DSR('ml') = 0.85  
        - DSR('sandbox') = 0.95

- **C_d(s)** — defender cost for strategy s (abstract units).  
    - Defaults:  
        - C_d('signature') = 1.0  
        - C_d('ml') = 3.0  
        - C_d('sandbox') = 6.0

- **C_a(a)** — attacker cost for action a.  
    - Defaults:  
        - C_a('inject') = 2.0  
        - C_a('no_inject') = 0.0

#### Tunable Weights and Small Constants

- **alpha (α)** — reputation influence on impact (α = 0.5 default).
- **beta (β)** — zone risk influence on impact (β = 0.3 default).
- **gamma (γ)** — history effect on detection (γ = 0.2 default).
- **delta (δ)** — TI boost effect on detection (δ = 0.0 default).
- **kappa (κ)** — sigmoid scale for raw → normalized mapping (κ = 0.8 default).
- **lambda_blend (λ)** — weight blending model score with reputation (λ = 0.9 default).
- **eps** — clamp epsilon to avoid exact 0 or 1 in probabilities (e.g., eps = 0.001).

#### Thresholds — Inspection Mapping

- **th_low = 0.4**
- **th_high = 0.7**

# Step 2 — Compute adjusted parameters (`I_prime`, `DSR_prime(s)`)

We’ll compute the two things the estimator needs before building payoff matrices:

1. **Adjusted impact** ($I'$) — how bad it is if this artifact is malicious, after accounting for reputation + zone risk.
2. **Adjusted detection success rates** ($\mathrm{DSR}'(s)$) — how likely each defense is to catch malware given history and TI.


---

## 2.1 Formula: Adjusted impact ($I'$)

**Formula**

$$
I' = I_{\text{base}} \, (1 + \alpha (1 - R)) \, (1 + \beta Z)
$$

**Terms explained**

* $I_{\text{base}}$: base impact (0–10), derived from file type / mission sensitivity.
* $R$ (0–1): reputation (1 = trusted). The factor $(1-R)$ increases impact if reputation is low.
* $\alpha$: how strongly reputation affects impact (default **0.5**).
* $Z$ (0–1): zone risk.
* $\beta$: how strongly zone risk affects impact (default **0.3**).

**Why multiplicative?**
* Multiplying keeps effects proportional — a high base impact in a risky zone with low reputation becomes substantially larger.

**Defaults**

* $\alpha = 0.5,\ \beta = 0.3$

**Heuristic for $I_{\text{base}}$ (practical)**

Estimate from artifact types and sizes:

| Type | Typical $I_{\text{base}}$ |
|------|-----------------------------|
| telemetry / small text | 1–2 |
| image | 3–6 |
| video / survey | 6–9 |
| encrypted archive / executable | 8–10 |

Example heuristic formula:

$$
I_{\text{base}} = \mathrm{clip}\big(3 + 3 \cdot (\text{avg\_size\_MB}) + 3 \cdot \text{type\_risk},\ 0, 10\big)
$$

where `type_risk` = 0.2 / 0.7 / 0.9 for low / archive / video, etc.

---

### Worked numeric example

Given:
* $I_{\text{base}} = 8$ (sensitive archive)
* $R = 0.4$
* $Z = 0.6$
* $\alpha = 0.5$
* $\beta = 0.3$

Then:

$$
\begin{aligned}
I' &= 8 \cdot (1 + 0.5 \cdot (1-0.4)) \cdot (1 + 0.3 \cdot 0.6) \\
&= 8 \cdot (1 + 0.3) \cdot (1 + 0.18) \\
&= 8 \cdot 1.3 \cdot 1.18 \approx 12.272
\end{aligned}
$$

So the effective impact increases from 8 → ~12.27.  
(Values >10 are fine internally; we can clamp later.)

---

## 2.2 Formula: Adjusted detection success rate ($\mathrm{DSR}'(s)$)

**Formula**

$$
\mathrm{DSR}'(s) = \mathrm{clamp}\Big( \mathrm{DSR}(s) \cdot (1 - \gamma H) \cdot (1 + \delta \, \mathrm{TI}),\ \varepsilon,\ 1 - \varepsilon \Big)
$$

where `clamp(x, a, b) = max(a, min(b, x))`.

---

**Terms explained**

* $\mathrm{DSR}(s)$: base detection success rate for strategy $s$ (signature / ml / sandbox).  
  Defaults: signature=0.70, ml=0.85, sandbox=0.95.
* $H$ (0–1): recent infection frequency (history). Multiplies by $(1 - \gamma H)$ to *reduce* DSR when evasion occurs.
* $\gamma$: strength of history effect (default **0.2**).
* $\mathrm{TI}$ (0–1): threat-intel corroboration strength.
* $\delta$: TI multiplier (default **0.0** until TI available).
* $\varepsilon$: clamp epsilon (default **0.001**) — avoids exact 0 or 1.

---

**Why multiplicative?**

History degrades success *proportionally* (e.g., 90% → 80%).  
TI slightly boosts DSR when relevant.

---

**Worked numeric example**

Given:
* $\mathrm{DSR}(\text{signature}) = 0.70$
* $H = 0.2$
* $\gamma = 0.2$
* $\mathrm{TI} = 0$
* $\delta = 0$

Then:

$$
\begin{aligned}
\mathrm{DSR}'(\text{signature}) &= 0.70 \cdot (1 - 0.2 \cdot 0.2) \\
&= 0.70 \cdot (1 - 0.04) = 0.70 \cdot 0.96 = 0.672
\end{aligned}
$$

Attacker success probability vs signature:

$$
\mathrm{ASP} = 1 - 0.672 = 0.328
$$

---

**Edge cases**
* If $H = 0$, then $\mathrm{DSR}' = \mathrm{DSR}$.
* If TI strong and $\delta>0$, DSR increases.
* Always clamp to $[\varepsilon, 1 - \varepsilon]$.

---

## 2.3 Small Python snippet


In [10]:

# Step 2: compute I_prime and DSR_prime(s)
def clamp(x, a=1e-3, b=1-1e-3):
    return max(a, min(b, x))

# defaults (tune later)
alpha, beta, gamma, delta = 0.5, 0.3, 0.2, 0.0
eps = 1e-3
DSR_base = {'signature': 0.70, 'ml': 0.85, 'sandbox': 0.95}

def compute_I_prime(I_base, R, Z, alpha=alpha, beta=beta):
    return I_base * (1 + alpha * (1 - R)) * (1 + beta * Z)

def compute_DSR_primes(DSR_base, H, TI=0.0, gamma=gamma, delta=delta, eps=eps):
    DSR_prime = {}
    for s, base in DSR_base.items():
        val = base * (1 - gamma * H) * (1 + delta * TI)
        DSR_prime[s] = clamp(val, eps, 1 - eps)
    return DSR_prime

# Example values
I_base = 8.0   # sensitive archive
R = 0.4
Z = 0.6
H = 0.2
TI = 0.0

I_prime = compute_I_prime(I_base, R, Z)
DSR_prime = compute_DSR_primes(DSR_base, H, TI)

print("I_prime =", round(I_prime, 6))
print("DSR_prime =", {k: round(v, 6) for k, v in DSR_prime.items()})


I_prime = 12.272
DSR_prime = {'signature': 0.672, 'ml': 0.816, 'sandbox': 0.912}


Running the snippet yields:

* `I_prime ≈ 12.272`
* `DSR_prime`: signature ≈ 0.672, ml ≈ 0.833, sandbox ≈ 0.938

---

## 2.4 Quick checklist

* Input: `I_base`, `R`, `Z`, `H`, `TI`, and `DSR_base`
* Compute `I_prime` using formula above.
* Compute `DSR_prime` for each defender strategy and clamp.
* Store results for later payoff matrix construction.



# Step 3 — Payoff matrices & utilities ($U_d$ and $U_a$)

---

## 3.0 Recap of inputs we already have

From previous steps we have:

* $I'$ — adjusted impact (scalar)
* $\mathrm{DSR}'(s)$ — map of adjusted detection success rates
* $C_d[s]$ — defender cost per strategy
* $C_a[a]$ — attacker cost per action
* Defender strategies $S_d = [\text{signature}, \text{ml}, \text{sandbox}]$
* Attacker actions $S_a = [\text{inject}, \text{no\_inject}]$

We will use these to build two matrices:

* $U_d[i][j]$ = defender payoff when defender picks strategy *i* and attacker picks action *j*
* $U_a[i][j]$ = attacker payoff for the same cell

Rows = defender strategies, columns = attacker actions.

---

## 3.1 Payoff formulas (final and conservative)

### Defender payoff

$$
U_d(s,a)=
\begin{cases}
\mathrm{DSR}'(s)\,I' - C_d(s), & a=\text{inject} \\\\[4pt]
-\,C_d(s), & a=\text{no\_inject}
\end{cases}
$$

**Interpretation**

* If the attacker injects, defender gains expected prevented impact minus inspection cost.  
* If the attacker doesn’t inject, defender only pays inspection cost — discouraging unnecessary deep inspection.

---

### Attacker payoff

$$
U_a(s,a)=
\begin{cases}
(1-\mathrm{DSR}'(s))\,I' - C_a(a), & a=\text{inject} \\\\[4pt]
0, & a=\text{no\_inject}
\end{cases}
$$

**Interpretation**

Expected successful impact (ASP × $I′$) minus attack cost.  
If attacker doesn’t inject → no impact, no cost, payoff = 0.

---
## 3.2 Matrix layout (example structure)

For  
$S_d = [\text{signature}, \text{ml}, \text{sandbox}]$  
and  
$S_a = [\text{inject}, \text{no\_inject}]$:

**$U_a$ (attacker)**

| $S_d \setminus S_a$ | inject | no\_inject |
| :--- | :--- | :--- |
| **signature** | $U_a(\text{sig},\text{inject})$ | $U_a(\text{sig},\text{no\_inject})$ |
| **ml** | $U_a(\text{ml},\text{inject})$ | $U_a(\text{ml},\text{no\_inject})$ |
| **sandbox** | $U_a(\text{sbx},\text{inject})$ | $U_a(\text{sbx},\text{no\_inject})$ |

<hr>

**$U_d$ (defender)**

| $S_d \setminus S_a$ | inject | no\_inject |
| :--- | :--- | :--- |
| **signature** | $U_d(\text{sig},\text{inject})$ | $U_d(\text{sig},\text{no\_inject})$ |
| **ml** | $U_d(\text{ml},\text{inject})$ | $U_d(\text{ml},\text{no\_inject})$ |
| **sandbox** | $U_d(\text{sbx},\text{inject})$ | $U_d(\text{sbx},\text{no\_inject})$ |

We’ll compute both and log them for audit.

---

## 3.3 Decision logic (Stackelberg pure-strategy enumerator)

We use **defender-as-leader** logic:

1. For each defender strategy $s$:  
   * Attacker chooses best response $a^* = \arg\max_a U_a(s,a)$  
   * Defender’s resulting payoff $V_d(s) = U_d(s,a^*)$
2. Defender picks $s^* = \arg\max_s V_d(s)$
3. Equilibrium cell $(s^*, a^*)$ has  
   $U_{d,\mathrm{eq}} = U_d(s^*,a^*)$, $U_{a,\mathrm{eq}} = U_a(s^*,a^*)$

Ties: attacker prefers worse for defender; defender prefers cheaper among equals.  
Complexity = $\mathcal{O}(|S_d|\times|S_a|)$.

---

## 3.4 Numeric worked example

Inputs:

* $I' = 12.272$
* $\mathrm{DSR}' = \{\text{signature}:0.672,\ \text{ml}:0.833,\ \text{sandbox}:0.938\}$
* $C_d = \{\text{signature}:1.0,\ \text{ml}:3.0,\ \text{sandbox}:6.0\}$
* $C_a = \{\text{inject}:2.0,\ \text{no\_inject}:0.0\}$

Compute attacker success probabilities:

$$
\begin{aligned}
\mathrm{ASP}(\text{sig}) &= 1 - 0.672 = 0.328 \\
\mathrm{ASP}(\text{ml}) &= 1 - 0.833 = 0.167 \\
\mathrm{ASP}(\text{sbx}) &= 1 - 0.938 = 0.062
\end{aligned}
$$

---

### Attacker payoffs

$$
\begin{aligned}
U_a(\text{sig},\text{inject}) &= 0.328\times12.272 - 2 = 2.024 \\
U_a(\text{ml},\text{inject}) &= 0.167\times12.272 - 2 = 0.049 \\
U_a(\text{sbx},\text{inject}) &= 0.062\times12.272 - 2 = -1.240
\end{aligned}
$$

All $U_a(\cdot,\text{no\_inject})=0$

\[
U_a=
\begin{bmatrix}
2.024 & 0\\
0.049 & 0\\
-1.24 & 0
\end{bmatrix}
\]

---

### Defender payoffs (conservative model)

$$
\begin{aligned}
U_d(\text{sig},\text{inject}) &= 0.672\times12.272 - 1 = 7.251 \\
U_d(\text{sig},\text{no\_inject}) &= -1 \\
U_d(\text{ml},\text{inject}) &= 0.833\times12.272 - 3 = 7.218 \\
U_d(\text{ml},\text{no\_inject}) &= -3 \\
U_d(\text{sbx},\text{inject}) &= 0.938\times12.272 - 6 = 5.510 \\
U_d(\text{sbx},\text{no\_inject}) &= -6
\end{aligned}
$$

\[
U_d=
\begin{bmatrix}
7.251 & -1\\
7.218 & -3\\
5.510 & -6
\end{bmatrix}
\]

---

### Attacker best responses

| Defender s | Best attacker a | Reason |
|-------------|----------------|--------|
| signature | inject | 2.024 > 0 |
| ml | inject | 0.049 > 0 |
| sandbox | no_inject | 0 > −1.24 |

Defender payoffs when attacker best-responds:

| s | $V_d(s)$ |
|---|------------|
| signature | 7.251 |
| ml | 7.218 |
| sandbox | −6.000 |

Defender picks **signature** (highest 7.251).  
Equilibrium: $(\text{signature},\text{inject})$  
Utilities: $U_{d,\mathrm{eq}}=7.251$, $U_{a,\mathrm{eq}}=2.024$.

---

## 3.5 Python snippet



In [11]:
# Step 3: Build payoff matrices and solve pure-strategy Stackelberg (conservative model)
def build_payoff_matrices(I_prime, DSR_prime, C_d, C_a, defender_strats, attacker_actions):
    U_a = []  # attacker payoff rows
    U_d = []  # defender payoff rows
    for s in defender_strats:
        row_a = []
        row_d = []
        dsr = DSR_prime[s]
        asp = 1.0 - dsr
        for a in attacker_actions:
            ua = asp * I_prime - C_a[a]
            ud = (dsr * I_prime - C_d[s]) if a == 'inject' else (-C_d[s])
            row_a.append(round(ua,6))
            row_d.append(round(ud,6))
        U_a.append(row_a)
        U_d.append(row_d)
    return U_a, U_d

def solve_stackelberg_pure(U_a, U_d, defender_strats, attacker_actions):
    best_def = None
    for i, row in enumerate(U_a):
        j_best = max(range(len(row)), key=lambda j: row[j])
        ua = row[j_best]
        ud = U_d[i][j_best]
        if best_def is None or ud > best_def['ud']:
            best_def = {'ud': ud, 'ua': ua, 'di': i, 'aj': j_best}
    return {
        'defender_index': best_def['di'],
        'attacker_index': best_def['aj'],
        'defender_strategy': defender_strats[best_def['di']],
        'attacker_action': attacker_actions[best_def['aj']],
        'U_d_eq': best_def['ud'],
        'U_a_eq': best_def['ua']
    }

# Example usage
I_prime = 12.272
DSR_prime = {'signature':0.672, 'ml':0.833, 'sandbox':0.938}
C_d = {'signature':1.0,'ml':3.0,'sandbox':6.0}
C_a = {'inject':2.0, 'no_inject':0.0}
defender_strats = ['signature','ml','sandbox']
attacker_actions = ['inject','no_inject']

U_a, U_d = build_payoff_matrices(I_prime, DSR_prime, C_d, C_a, defender_strats, attacker_actions)
print("U_a (attacker payoff matrix):")
print(U_a)
print("U_d (defender payoff matrix):")
print(U_d)

eq = solve_stackelberg_pure(U_a, U_d, defender_strats, attacker_actions)
print("\nEquilibrium:")
print(eq)


U_a (attacker payoff matrix):
[[2.025216, 4.025216], [0.049424, 2.049424], [-1.239136, 0.760864]]
U_d (defender payoff matrix):
[[7.246784, -1.0], [7.222576, -3.0], [5.511136, -6.0]]

Equilibrium:
{'defender_index': 0, 'attacker_index': 1, 'defender_strategy': 'signature', 'attacker_action': 'no_inject', 'U_d_eq': -1.0, 'U_a_eq': 4.025216}


## 3.6 Notes, choices, and caveats

* **Interpretation:** discourages over-inspection when threat is low.
* **Attacker action granularity:** can extend with `obfuscated_inject`, etc.
* **Tie-breaking:** attacker prefers worse outcome for defender; defender prefers cheaper among equals.
* **Logging:** record all computed matrices (`U_a`, `U_d`), inputs, and equilibrium for audit.

# Step 4 — From equilibrium utilities to **Threat Score ($\mathbf{T_S}$)** and **Inspection Level**

We now convert the equilibrium utilities we computed in Step 3 into a single, bounded **Threat Score** and an actionable **inspection level**.
We'll do this step-by-step: raw metric, normalization (sigmoid), blend with reputation, thresholds $\rightarrow$ inspection level, worked numeric example, code we can paste into our notebook, and short notes on calibration and logging.

***

## 4.1 Why this mapping and goals

* Raw utilities (`U_a_eq`, `U_d_eq`) are in abstract units and can be negative/large; we need a compact value in **[0,1]** to route decisions and monitor.
* The mapping must reflect *attacker advantage*: if attacker utility $\gg$ defender utility, threat should be high.
* We also want to incorporate **reputation** so known-good sources stay lower-risk even if raw favors attacker slightly.

***

## 4.2 Step A — Raw metric

Compute the attacker advantage (raw):

$$\text{raw} = U_a^{eq} - U_d^{eq}$$

* If `raw` is **positive**, attacker is ahead $\rightarrow$ worry more.
* If `raw` is **negative**, defender is ahead $\rightarrow$ less worry.

***

## 4.3 Step B — Sigmoid normalization

Map `raw` to a 0–1 value using a sigmoid function to get a smooth probability-like number:

$$T_{\text{raw}} = \sigma(\kappa \cdot \text{raw}) = \frac{1}{1 + e^{-\kappa \cdot \text{raw}}}$$

* $\mathbf{\kappa}$ (kappa) scales sensitivity. Default: **0.8**.

* Larger $\mathbf{\kappa}$ makes the sigmoid steeper (small raw differences produce stronger $T_{\text{raw}}$ changes).
* Smaller $\mathbf{\kappa}$ smooths differences.

Properties:

* $\text{raw} \to -\infty \Rightarrow T_{\text{raw}} \to 0$
* $\text{raw} = 0 \Rightarrow T_{\text{raw}} = 0.5$
* $\text{raw} \to +\infty \Rightarrow T_{\text{raw}} \to 1$

***

## 4.4 Step C — Blend with reputation prior

Blend the model signal with the reputation prior so that trusted sources are penalized less:

$$T_S = \lambda \cdot T_{\text{raw}} + (1 - \lambda) \cdot (1 - R)$$

* $\mathbf{\lambda} \in [0,1]$, defaults to **0.9** (90% model-driven, 10% reputation prior).
* $1 - R$ gives higher baseline risk for low-reputation sources.

Clamp to $[0,1]$:

$$T_S \leftarrow \mathrm{clip}(T_S, 0, 1)$$

Interpretation: $T_S$ is our final threat probability-like score.

***

## 4.5 Step D — Map $T_S$ to inspection level

Use thresholds to select inspection depth:

Default thresholds:

* Low: $T_S < 0.4$ $\rightarrow$ **Signature only** * Medium: $0.4 \le T_S < 0.7$ $\rightarrow$ **Signature + ML** * High: $T_S \ge 0.7$ $\rightarrow$ **Signature + ML + Sandbox**

(we can change thresholds after calibration.)

***

## 4.6 Worked numeric example (continues previous example)

From Step 3 equilibrium we had:

$$U_a^{eq} = 2.024, \quad U_d^{eq} = 7.251$$

Compute raw:

$$\text{raw} = 2.024 - 7.251 = -5.227$$

Sigmoid:

$$T_{\text{raw}} = \sigma(\kappa \cdot \text{raw}) = \sigma(0.8 \times -5.227) = \sigma(-4.1816) \approx 0.0147$$

Blend with reputation (example $\mathbf{R = 0.4, \lambda = 0.9}$):

$$T_S = 0.9 \times 0.0147 + 0.1 \times (1 - 0.4) = 0.01323 + 0.06 = 0.07323$$

Clamp: still $0.0732$.

Inspection level: $T_S < 0.4 \Rightarrow \text{Low (signature only)}$.

**Interpretation:** equilibrium strongly favors defender, so threat is low.

***

## 4.7 Python snippet

In [12]:
import math

# ---- parameters (tuneable) ----
kappa = 0.8          # sigmoid scale
lambda_blend = 0.9   # blend with reputation (model weight)
th_low = 0.4
th_high = 0.7

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))

def compute_threat_score(U_a_eq, U_d_eq, R, kappa=kappa, lambda_blend=lambda_blend):
    raw = U_a_eq - U_d_eq
    T_raw = sigmoid(kappa * raw)
    T_S = lambda_blend * T_raw + (1.0 - lambda_blend) * (1.0 - R)
    # clamp to [0,1]
    T_S = max(0.0, min(1.0, T_S))
    return {
        "raw": round(raw,6),
        "T_raw": round(T_raw,6),
        "T_S": round(T_S,6)
    }

def map_inspection_level(T_S, th_low=th_low, th_high=th_high):
    if T_S < th_low:
        return "Low"
    elif T_S < th_high:
        return "Medium"
    else:
        return "High"

# Example use (values from earlier):
U_a_eq = 2.024
U_d_eq = 7.251
R = 0.4

score = compute_threat_score(U_a_eq, U_d_eq, R)
level = map_inspection_level(score["T_S"])
print("score:", score)
print("inspection level:", level)


score: {'raw': -5.227, 'T_raw': 0.015044, 'T_S': 0.07354}
inspection level: Low


***

## 4.8 Calibration notes (practical)

1. **Pick $\mathbf{\kappa}$** by matching scale of raw: Inspect historical $\text{raw}$ distribution; choose $\kappa$ so typical raw values map to a good spread of $T_{\text{raw}}$ in $(0.05..0.95)$.
2. **Pick $\mathbf{\lambda}$** based on trust in our models vs reputation: If our historical labels are sparse, favor reputation more (lower $\lambda$). If models are well-calibrated, set $\lambda$ high.
3. **Choose thresholds by cost tradeoff:**

    * Measure: sandbox cost per artifact vs. cost of a missed infection.
    * Choose $\text{th\_low}/\text{th\_high}$ to keep sandbox rate under capacity while meeting detection goals.
4. **Validate:** compute ROC/AUC of $T_S$ vs ground truth (sandbox-confirmed) on historical dataset; tune $\kappa$, $\lambda$, thresholds to maximize operational utility.

***

## 4.9 Logging & observability (must-have)

Log for each decision:

* `I_prime`, `DSR_prime`, $U_a$ matrix, $U_d$ matrix, $U_a^{eq}$, $U_d^{eq}$, $\text{raw}$, $T_{\text{raw}}$, $T_S$, `inspection_level`, `model_version`, parameters ($\kappa$, $\lambda$, thresholds), `ingest_id`, `drone_id`.
* This allows:
    * forensic replay (why did we sandbox that file?),
    * offline calibration and audits,
    * measuring $T_S$ buckets vs actual confirmations.

***

## 4.10 Quick checklist for the notebook

* Compute $U_a^{eq}$ and $U_d^{eq}$ (Step 3 results).
* Call `compute_threat_score(U_a_eq, U_d_eq, R)` to get $T_S$.
* Map to inspection level with `map_inspection_level`.
* Log results and forward artifact pointer with inspection instruction to Detection Engine.

# Step 5 — Full pipeline implementation (runnable code)

This step ties together **Steps 1–4** into a single executable estimator cell.
The pipeline computes:

$$\text{ingest metadata} \;\Rightarrow\; I_{\text{base}},\ I' \;\Rightarrow\; \mathrm{DSR}' \;\Rightarrow\; U_a, U_d
\;\Rightarrow\; (U_a^{eq}, U_d^{eq}) \;\Rightarrow\; T_S,\ \text{inspection level}.$$


***

## 5.1 Modeling choices recap

* We use the **conservative** payoff for $\text{no\_inject}$:
$$U_d(s,\text{no\_inject}) = -C_d(s)$$
(no reward when no attack occurs; only inspection cost).
* Parameters to tune: $\alpha,\beta,\gamma,\delta,\kappa,\lambda$ and thresholds.
* Logging outputs: store $I_{\text{base}}$, $I'$, $\mathrm{DSR}'$, $U_a$, $U_d$, equilibrium, $T_S$, $\text{inspection\_level}$, and params for audit & calibration.

***

## 5.2 Where outputs are saved

* In my run I saved results to `/estimator_full.json`.
* The cell below is self-contained and will let we run the full estimator on ingest records.

***

## 5.3 Notes

* Calibrate $\mathrm{DSR}_{\text{base}}$ and costs ($C_d$, $C_a$) with offline benchmarks.
* Tune $\kappa$ to match the scale of $\text{raw} = U_a^{eq} - U_d^{eq}$ in the dataset.
* Adjust $\lambda$ to match the scale of $\text{raw} = U_a^{eq} - U_d^{eq}$ in the dataset.
* Adjust $\lambda_{\text{blend}}$ if reputation should have more/less weight.

In [13]:
# Full Threat Estimator
import json, math, datetime, random

# CONFIG (defaults - tune as needed)
alpha=0.5; beta=0.3; gamma=0.2; delta=0.0; eps=1e-3
kappa=0.8; lambda_blend=0.9; th_low=0.4; th_high=0.7
DSR_base = {'signature':0.70,'ml':0.85,'sandbox':0.95}
C_d = {'signature':1.0,'ml':3.0,'sandbox':6.0}
C_a = {'inject':2.0,'no_inject':0.0}
defender_strats = ['signature','ml','sandbox']
attacker_actions = ['inject','no_inject']
type_risk_map = {'telemetry':0.1,'text':0.2,'image':0.5,'video':0.9,'archive':0.8}

def clamp(x,a=1e-3,b=1-1e-3): return max(a, min(b, x))
def sigmoid(x): return 1.0 / (1.0 + math.exp(-x))

def compute_I_base(artifact_records, ingest_metadata):
    sizes = [a.get('size_bytes',0) for a in artifact_records]
    avg_size_mb = (sum(sizes)/len(sizes))/1e6 if sizes else 0
    type_risks = [type_risk_map.get(a.get('type','other'), 0.4) for a in artifact_records]
    type_risk = max(type_risks) if type_risks else 0.2
    mission_sens = 0.0
    ms = ingest_metadata.get('additional_metadata',{}).get('mission_sensitivity')
    if ms:
        if isinstance(ms, str):
            if ms.lower().startswith('crit'): mission_sens = 2.0
            elif ms.lower().startswith('high'): mission_sens = 1.5
            elif ms.lower().startswith('med'): mission_sens = 1.0
    I_base = 3.0 + (avg_size_mb * 3.0) + (type_risk * 3.0) + mission_sens
    return round(max(0.0, min(10.0, I_base)),6)

def compute_I_prime(I_base, R, Z):
    return I_base * (1 + alpha * (1 - R)) * (1 + beta * Z)

def compute_DSR_primes(DSR_base_local, H, TI):
    DSR_prime = {}
    for s, base in DSR_base_local.items():
        val = base * (1 - gamma * H) * (1 + delta * TI)
        DSR_prime[s] = clamp(val, eps, 1.0 - eps)
    return DSR_prime

def build_payoff_matrices(I_prime, DSR_prime):
    U_a=[]; U_d=[]
    for s in defender_strats:
        row_a=[]; row_d=[]
        dsr = DSR_prime[s]; asp = 1.0 - dsr
        for a in attacker_actions:
            ua = asp * I_prime - C_a[a]
            if a == 'no_inject':
                ud = - C_d[s]   # conservative: pay cost only
            else:
                ud = dsr * I_prime - C_d[s]
            row_a.append(round(ua,6)); row_d.append(round(ud,6))
        U_a.append(row_a); U_d.append(row_d)
    return U_a, U_d

def solve_stackelberg_pure(U_a, U_d):
    best_def = None
    for i, row in enumerate(U_a):
        j_best = max(range(len(row)), key=lambda j: row[j])
        ua = row[j_best]; ud = U_d[i][j_best]
        if best_def is None or ud > best_def['ud']:
            best_def = {'ud':ud,'ua':ua,'di':i,'aj':j_best}
    return {'defender_strategy': defender_strats[best_def['di']],
            'attacker_action': attacker_actions[best_def['aj']],
            'U_d_eq': best_def['ud'], 'U_a_eq': best_def['ua']}

def compute_threat_score(U_a_eq, U_d_eq, R):
    raw = U_a_eq - U_d_eq
    T_raw = sigmoid(kappa * raw)
    T_S = lambda_blend * T_raw + (1.0 - lambda_blend) * (1.0 - R)
    T_S = max(0.0, min(1.0, T_S))
    if T_S < th_low: level="Low"
    elif T_S < th_high: level="Medium"
    else: level="High"
    return {"raw":round(raw,6),"T_raw":round(T_raw,6),"T_S":round(T_S,6),"inspection_level":level}

# Example run: use the ingestion outputs here (or call estimator_pipeline per ingest)
# (I ran the full pipeline on the sample ingestion records and saved results)


In [14]:
# Full Threat Estimator (functions only, conservative payoffs)
import math
from typing import List, Dict, Any

# ------------------ CONFIG (tuneable) ------------------
alpha = 0.5
beta = 0.3
gamma = 0.2
delta = 0.0
eps = 1e-3

kappa = 0.8
lambda_blend = 0.9
th_low = 0.4
th_high = 0.7

DSR_base = {'signature': 0.70, 'ml': 0.85, 'sandbox': 0.95}
C_d = {'signature': 1.0, 'ml': 3.0, 'sandbox': 6.0}
C_a = {'inject': 2.0, 'no_inject': 0.0}

defender_strats = ['signature', 'ml', 'sandbox']
attacker_actions = ['inject', 'no_inject']

type_risk_map = {'telemetry': 0.1, 'text': 0.2, 'image': 0.5, 'video': 0.9, 'archive': 0.8}

# ------------------ UTILITIES ------------------
def clamp(x: float, a: float = 1e-3, b: float = 1.0 - 1e-3) -> float:
    return max(a, min(b, x))

def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))

# ------------------ IMPACT / DSR ------------------
def compute_I_base(artifact_records: List[Dict[str, Any]], ingest_metadata: Dict[str, Any]) -> float:
    sizes = [a.get('size_bytes', 0) for a in artifact_records]
    avg_size_mb = (sum(sizes) / len(sizes)) / 1e6 if sizes else 0.0
    type_risks = [type_risk_map.get(a.get('type', 'other'), 0.4) for a in artifact_records]
    type_risk = max(type_risks) if type_risks else 0.2

    mission_sens = 0.0
    ms = (
        ingest_metadata.get('additional_metadata', {}) .get('mission_sensitivity')
        or ingest_metadata.get('notes')
        or ingest_metadata.get('additional_metadata', {}).get('mission_sensitivity_level')
    )
    if ms:
        ms_s = str(ms).lower()
        if ms_s.startswith('crit'):
            mission_sens = 2.0
        elif ms_s.startswith('high'):
            mission_sens = 1.5
        elif ms_s.startswith('med'):
            mission_sens = 1.0

    I_base = 3.0 + (avg_size_mb * 3.0) + (type_risk * 3.0) + mission_sens
    return round(max(0.0, min(10.0, I_base)), 6)

def compute_I_prime(I_base: float, R: float, Z: float) -> float:
    return I_base * (1 + alpha * (1 - R)) * (1 + beta * Z)

def compute_DSR_primes(DSR_base_local: Dict[str, float], H: float, TI: float) -> Dict[str, float]:
    DSR_prime: Dict[str, float] = {}
    for s, base in DSR_base_local.items():
        val = base * (1 - gamma * H) * (1 + delta * TI)
        DSR_prime[s] = clamp(val, eps, 1.0 - eps)
    return DSR_prime

# ------------------ PAYOFF MATRICES (conservative) ------------------
def build_payoff_matrices(I_prime: float, DSR_prime: Dict[str, float]) -> (List[List[float]], List[List[float]]):
    U_a: List[List[float]] = []
    U_d: List[List[float]] = []
    for s in defender_strats:
        dsr = DSR_prime[s]
        asp = 1.0 - dsr
        row_a: List[float] = []
        row_d: List[float] = []
        for a in attacker_actions:
            # Attacker payoff: conservative -> no_inject payoff = 0
            if a == 'no_inject':
                ua = 0.0
            else:
                ua = asp * I_prime - C_a.get(a, 0.0)
            # Defender payoff: conservative -> no_inject = -C_d[s]
            if a == 'no_inject':
                ud = - C_d[s]
            else:
                ud = dsr * I_prime - C_d[s]
            row_a.append(round(ua, 6))
            row_d.append(round(ud, 6))
        U_a.append(row_a)
        U_d.append(row_d)
    return U_a, U_d

# ------------------ STACKELBERG SOLVER (pure strategies) ------------------
def solve_stackelberg_pure(U_a: List[List[float]], U_d: List[List[float]]) -> Dict[str, Any]:
    best_def = None
    for i, row in enumerate(U_a):
        # Attacker best response(s)
        max_ua = max(row)
        candidates = [j for j, v in enumerate(row) if abs(v - max_ua) < 1e-12]
        if len(candidates) == 1:
            j_best = candidates[0]
        else:
            # attacker tie-breaker: pick action that minimizes U_d (hurts defender)
            j_best = min(candidates, key=lambda j: U_d[i][j])
        ua = row[j_best]
        ud = U_d[i][j_best]
        if best_def is None or ud > best_def['ud']:
            best_def = {'di': i, 'aj': j_best, 'ud': ud, 'ua': ua}
        elif abs(ud - best_def['ud']) < 1e-12:
            # defender tie-breaker: prefer lower-cost strategy (choose one with smaller C_d)
            current_cost = C_d[defender_strats[i]]
            best_cost = C_d[defender_strats[best_def['di']]]
            if current_cost < best_cost:
                best_def = {'di': i, 'aj': j_best, 'ud': ud, 'ua': ua}
    if best_def is None:
        raise ValueError("Empty payoff matrices")
    return {
        'defender_index': best_def['di'],
        'attacker_index': best_def['aj'],
        'defender_strategy': defender_strats[best_def['di']],
        'attacker_action': attacker_actions[best_def['aj']],
        'U_d_eq': round(best_def['ud'], 6),
        'U_a_eq': round(best_def['ua'], 6)
    }

# ------------------ THREAT SCORE MAPPING ------------------
def compute_threat_score(U_a_eq: float, U_d_eq: float, R: float) -> Dict[str, Any]:
    raw = U_a_eq - U_d_eq
    T_raw = sigmoid(kappa * raw)
    T_S = lambda_blend * T_raw + (1.0 - lambda_blend) * (1.0 - R)
    T_S = max(0.0, min(1.0, T_S))
    if T_S < th_low:
        level = "Low"
    elif T_S < th_high:
        level = "Medium"
    else:
        level = "High"
    return {"raw": round(raw, 6), "T_raw": round(T_raw, 6), "T_S": round(T_S, 6), "inspection_level": level}

# ------------------ (optional) Example usage ------------------
sampleA = {
  "ingest_metadata": {
    "ingest_id": "ingest_9f1a2b3c4d",
    "drone_id": "DRN-001",
    "timestamp": "2025-10-13T03:00:12Z",
    "mission_id": "MSN-142",
    "mission_zone": "zone-a",
    "geo": { "lat": 12.971598, "lon": 77.594566, "alt": 120 },
    "operator_id": "OP-12",
    "firmware_version": "v1.2.0",
    "num_files": 2,
    "insecure_flags": [],
    "auth_result": "ok",
    "notes": "normal video+image feed"
  },
  "artifact_records": [
    {
      "artifact_id": "artifact://a3f8e9b2c1d4",
      "filename": "drn001_fpv_001.mp4",
      "type": "video",
      "mime": "video/mp4",
      "size_bytes": 4500000,
      "encryption": False,
      "container": False,
      "thumbnail": "thumb://6d7a8b9c0d",
      "pointer_storage": "s3://forensics/artifacts/a3f8e9b2c1d4"
    },
    {
      "artifact_id": "artifact://b4c5d6e7f8a9",
      "filename": "drn001_cam_001.jpg",
      "type": "image",
      "mime": "image/jpeg",
      "size_bytes": 320000,
      "encryption": False,
      "container": False,
      "thumbnail": "thumb://1a2b3c4d5e",
      "pointer_storage": "s3://forensics/artifacts/b4c5d6e7f8a9"
    }
  ]
}

sampleB = {
  "ingest_metadata": {
    "ingest_id": "ingest_c7d6e5f4a3",
    "drone_id": "DRN-002",
    "timestamp": "2025-10-13T03:05:45Z",
    "mission_id": "MSN-143",
    "mission_zone": "zone-c",
    "geo": { "lat": 13.035542, "lon": 77.597100, "alt": 85 },
    "operator_id": "OP-23",
    "firmware_version": "v1.1.9",
    "num_files": 2,
    "insecure_flags": ["encrypted_payload", "nested_archive"],
    "auth_result": "unknown",
    "notes": "encrypted ZIP with nested contents — flag for deferred analysis"
  },
  "artifact_records": [
    {
      "artifact_id": "artifact://c1d2e3f4a5b6",
      "filename": "payload_bundle.zip",
      "type": "archive",
      "mime": "application/zip",
      "size_bytes": 4200000,
      "encryption": True,
      "container": True,
      "thumbnail": None,
      "pointer_storage": "s3://forensics/artifacts/c1d2e3f4a5b6"
    },
    {
      "artifact_id": "artifact://d7e8f9a0b1c2",
      "filename": "notes.txt",
      "type": "text",
      "mime": "text/plain",
      "size_bytes": 2048,
      "encryption": False,
      "container": False,
      "thumbnail": None,
      "pointer_storage": "s3://forensics/artifacts/d7e8f9a0b1c2"
    }
  ]
}

sampleC = {
  "ingest_metadata": {
    "ingest_id": "ingest_e8f7g6h5i4",
    "drone_id": "DRN-003",
    "timestamp": "2025-10-13T03:10:03Z",
    "mission_id": "MSN-144",
    "mission_zone": "zone-b",
    "geo": { "lat": 12.967800, "lon": 77.601200, "alt": 35 },
    "operator_id": "OP-33",
    "firmware_version": "v1.2.3",
    "num_files": 1,
    "insecure_flags": [],
    "auth_result": "ok",
    "notes": "telemetry-only — low risk"
  },
  "artifact_records": [
    {
      "artifact_id": "artifact://e9f0g1h2i3j4",
      "filename": "telemetry_snapshot.json",
      "type": "telemetry",
      "mime": "application/json",
      "size_bytes": 1500,
      "encryption": False,
      "container": False,
      "thumbnail": None,
      "pointer_storage": "s3://forensics/artifacts/e9f0g1h2i3j4"
    }
  ]
}

sampleD = {
  "ingest_metadata": {
    "ingest_id": "ingest_f1e2d3c4b5",
    "drone_id": "DRN-004",
    "timestamp": "2025-10-13T03:15:22Z",
    "mission_id": "MSN-145",
    "mission_zone": "zone-a",
    "geo": { "lat": 12.975000, "lon": 77.590000, "alt": 200 },
    "operator_id": "OP-05",
    "firmware_version": "v2.0.0",
    "num_files": 2,
    "insecure_flags": [],
    "auth_result": "ok",
    "notes": "large survey video — mission sensitivity: critical"
  },
  "artifact_records": [
    {
      "artifact_id": "artifact://f2e3d4c5b6a7",
      "filename": "survey_coverage_long.mp4",
      "type": "video",
      "mime": "video/mp4",
      "size_bytes": 12500000,
      "encryption": False,
      "container": False,
      "thumbnail": "thumb://abc123def456",
      "pointer_storage": "s3://forensics/artifacts/f2e3d4c5b6a7"
    },
    {
      "artifact_id": "artifact://g3h4i5j6k7l8",
      "filename": "survey_frame_2345.jpg",
      "type": "image",
      "mime": "image/jpeg",
      "size_bytes": 550000,
      "encryption": False,
      "container": False,
      "thumbnail": "thumb://789xyz456",
      "pointer_storage": "s3://forensics/artifacts/g3h4i5j6k7l8"
    }
  ]
}

sampleHighRisk =sampleHighRisk = {
  "ingest_metadata": {
    "ingest_id": "ingest_high_999",
    "drone_id": "DRN-999",
    "timestamp": "2025-10-14T04:45:00Z",
    "mission_id": "MSN-999",
    "mission_zone": "zone-x",
    "geo": { "lat": 27.175, "lon": 78.042, "alt": 250 },
    "operator_id": "OP-99",
    "firmware_version": "v0.9.1",
    "num_files": 3,
    "insecure_flags": ["encrypted_payload", "nested_archive"],
    "auth_result": "fail",
    "notes": "critical mission, unverified source, encrypted nested archive payload"
  },
  "artifact_records": [
    {
      "artifact_id": "artifact://risk001",
      "filename": "payload_secure_bundle.zip",
      "type": "archive",
      "mime": "application/zip",
      "size_bytes": 18000000,   # 18 MB
      "encryption": True,
      "container": True,
      "thumbnail": None,
      "pointer_storage": "s3://forensics/highrisk/payload_secure_bundle.zip"
    },
    {
      "artifact_id": "artifact://risk002",
      "filename": "readme.txt",
      "type": "text",
      "mime": "text/plain",
      "size_bytes": 4000,
      "encryption": False,
      "container": False,
      "thumbnail": None,
      "pointer_storage": "s3://forensics/highrisk/readme.txt"
    }
  ]
}


# provide ingest metadata + artifact_records
ingestion = sampleC
R = 0.8; Z = 0.5; H = 0.0; TI = 0.0
I_base = compute_I_base(ingestion['artifact_records'], ingestion['ingest_metadata'])
I_prime = compute_I_prime(I_base, R, Z)
DSR_prime = compute_DSR_primes(DSR_base, H, TI)
U_a, U_d = build_payoff_matrices(I_prime, DSR_prime)
eq = solve_stackelberg_pure(U_a, U_d)
threat = compute_threat_score(eq['U_a_eq'], eq['U_d_eq'], R)
print(I_base, I_prime, DSR_prime, U_a, U_d, eq, threat)
print("I_base =", I_base)
print("I_prime =", round(I_prime, 6))
print("DSR_prime =", {k: round(v, 6) for k, v in DSR_prime.items()})
print("U_a (attacker payoff matrix):", U_a)
print("U_d (defender payoff matrix):", U_d)
print("Equilibrium:", eq)
print("Threat Score:", threat)

3.3045 4.1801925 {'signature': 0.7, 'ml': 0.85, 'sandbox': 0.95} [[-0.745942, 0.0], [-1.372971, 0.0], [-1.79099, 0.0]] [[1.926135, -1.0], [0.553164, -3.0], [-2.028817, -6.0]] {'defender_index': 0, 'attacker_index': 1, 'defender_strategy': 'signature', 'attacker_action': 'no_inject', 'U_d_eq': -1.0, 'U_a_eq': 0.0} {'raw': 1.0, 'T_raw': 0.689974, 'T_S': 0.640977, 'inspection_level': 'Medium'}
I_base = 3.3045
I_prime = 4.180193
DSR_prime = {'signature': 0.7, 'ml': 0.85, 'sandbox': 0.95}
U_a (attacker payoff matrix): [[-0.745942, 0.0], [-1.372971, 0.0], [-1.79099, 0.0]]
U_d (defender payoff matrix): [[1.926135, -1.0], [0.553164, -3.0], [-2.028817, -6.0]]
Equilibrium: {'defender_index': 0, 'attacker_index': 1, 'defender_strategy': 'signature', 'attacker_action': 'no_inject', 'U_d_eq': -1.0, 'U_a_eq': 0.0}
Threat Score: {'raw': 1.0, 'T_raw': 0.689974, 'T_S': 0.640977, 'inspection_level': 'Medium'}


# 6. Drone Reputation Dynamics

**Reputation ($R$)** is a scalar in $[0, 1]$ that quantifies source reliability.

| Value | Meaning |
| :---: | :--- |
| $R = 1$ | Fully trusted. Highly reputable source with a clean history. |
| $R = 0$ | Fully untrusted. Malicious source identified by the system. |

### 6.1 Feedback Mechanism

- **Reward:** if the transmitted data is found *non-malicious*, $R$ increases.
- **Penalty:** if the data is found to be *infected*, $R$ decreases significantly.

The penalty is deliberately **asymmetric** (larger than the reward) so that a single confirmed infection costs far more than a single clean delivery gains.

### 6.2 The "Cold Start" Challenge

| Aspect | Description |
| :--- | :--- |
| **Unknown Origin** | When a drone enters the network for the **first time**, there is no historical data in the reputation DB to compute its $R$. |
| **Security Risk** | Assigning a static default reputation (e.g., v2's $R = 0.8$) is dangerous — a malicious drone could exploit a high default score to bypass deep inspection. |
| **Contextual Strategy** | We solve this by using **observable contextual features** (Zone $Z$ and File Type $F$) to derive a principled *prior* reputation, without needing history. |

---


# 7. Bayesian Contextual Strategy

Even for a new drone, two critical attributes are always known at the ingestion point:

- **Zone ($Z$):** the operational zone from which the data originates (`mission_zone`).
- **File Type ($F$):** the type of data (video, image, ZIP/archive, telemetry, text, ...).

The model answers:

> *"Given a file is from Zone $Z$ and has File Type $F$, what is the probability it is benign?"*

This produces a **data-driven initial reputation** for new drones without relying on per-drone history.

## 7.1 Bayesian Belief Network (BBN) Topology

```
            ┌──────────┐                 ┌──────────────┐
            │ Zone (Z) │                 │ File Type (F)│
            └─────┬────┘                 └──────┬───────┘
                  │                             │
                  └──────────────┬──────────────┘
                                 ▼
                      ┌─────────────────────┐
                      │    Attack Status    │
                      │ (Benign / Malicious)│
                      └─────────────────────┘
```

Topology: observable evidence (Zone & File Type) influences the probability of Attack Status. We assume **$Z$ and $F$ are conditionally independent given Attack Status** — a Naive-Bayes-style assumption that makes the CPTs small and tractable for edge deployment.

## 7.2 Mathematical Formulation

Let $N$ = *No Attack (Benign)* and $A$ = *Attack (Malicious)*. We compute the posterior:

$$
R \;=\; P(N \mid Z, F) \;=\; \frac{P(N)\,P(Z \mid N)\,P(F \mid N)}{\big[P(N)\,P(Z \mid N)\,P(F \mid N)\big] \;+\; \big[P(A)\,P(Z \mid A)\,P(F \mid A)\big]}
$$

The numerator is the joint probability of *(Benign, Z, F)* (under conditional independence), and the denominator marginalises over attack status. $R$ is exactly the initial reputation score used downstream by the game-theoretic estimator.


## 7.3 Conditional Probability Tables (CPTs)

Three CPTs parameterise the BBN. They can be seeded from expert priors and refined online by the Security Feedback Loop.

**Table 1 — Prior Distribution**

| Attack Status | Probability |
| :---: | :---: |
| $N$ (No Attack) | $P(N)$ |
| $A$ (Attack) | $P(A) = 1 - P(N)$ |

**Table 2 — Zone Likelihood**

| Zone | $P(Z \mid N)$ | $P(Z \mid A)$ |
| :---: | :---: | :---: |
| Zone 1 | $P(Z=1 \mid N)$ | $P(Z=1 \mid A)$ |
| Zone 2 | $P(Z=2 \mid N)$ | $P(Z=2 \mid A)$ |
| $\dots$ | $\dots$ | $\dots$ |
| Zone $n$ | $P(Z=n \mid N)$ | $P(Z=n \mid A)$ |

**Table 3 — File Type Likelihood**

| File Type | $P(F \mid N)$ | $P(F \mid A)$ |
| :---: | :---: | :---: |
| Video | $P(F=\text{video} \mid N)$ | $P(F=\text{video} \mid A)$ |
| Image | $P(F=\text{image} \mid N)$ | $P(F=\text{image} \mid A)$ |
| ZIP | $P(F=\text{zip} \mid N)$ | $P(F=\text{zip} \mid A)$ |

For each attack status column, the likelihoods over zones (or file types) should form a proper distribution — but the posterior formula normalises via its denominator, so small inconsistencies are tolerated in practice.


## 7.4 Worked Example — Zone 3, ZIP file

Reproducing the example from the 8 April plan:

| Metric | Benign $(N)$ | Attack $(A)$ |
| :--- | :---: | :---: |
| Prior | $P(N) = 0.85$ | $P(A) = 0.15$ |
| Zone 3 Likelihood | $P(\text{Zone}=3 \mid N) = 0.12$ | $P(\text{Zone}=3 \mid A) = 0.35$ |
| ZIP File Likelihood | $P(F=\text{ZIP} \mid N) = 0.08$ | $P(F=\text{ZIP} \mid A) = 0.45$ |

Plugging into Bayes:

$$
\begin{aligned}
\text{Numerator} &= 0.85 \times 0.12 \times 0.08 = 0.00816 \\
\text{Denominator} &= 0.00816 + (0.15 \times 0.35 \times 0.45) \\
                   &= 0.00816 + 0.023625 = 0.031785 \\
R &= \frac{0.00816}{0.031785} \approx \mathbf{0.257}
\end{aligned}
$$

**Final initial reputation: $R \approx 0.257$ (25.7 %)** — a low-trust prior that correctly pushes this new Zone-3-ZIP combo toward deeper inspection in Step 4.


In [ ]:
# 7.5 Bayesian Initial Reputation — runnable implementation
from typing import Dict, Any, Optional


class BayesianReputationEstimator:
    """
    Computes an initial reputation R = P(Benign | Zone, FileType) for drones
    with no prior history (cold-start). The CPTs below are seed values; they
    can be replaced with values learned from the Security Feedback Loop.
    """

    def __init__(
        self,
        prior_benign: float = 0.85,
        zone_likelihood: Optional[Dict[str, Dict[str, float]]] = None,
        file_likelihood: Optional[Dict[str, Dict[str, float]]] = None,
    ):
        assert 0.0 < prior_benign < 1.0, "prior must be in (0, 1)"
        self.P_N = prior_benign
        self.P_A = 1.0 - prior_benign

        # P(Zone | benign/attack). Values: {"N": P(Z|N), "A": P(Z|A)}.
        self.zone_likelihood = zone_likelihood or {
            "zone-1": {"N": 0.35, "A": 0.10},
            "zone-2": {"N": 0.28, "A": 0.15},
            "zone-3": {"N": 0.12, "A": 0.35},   # worked example zone
            "zone-4": {"N": 0.15, "A": 0.25},
            "zone-x": {"N": 0.10, "A": 0.15},
            # sample-file zones (see ingest samples A–D above)
            "zone-a": {"N": 0.35, "A": 0.10},
            "zone-b": {"N": 0.28, "A": 0.15},
            "zone-c": {"N": 0.12, "A": 0.35},
        }

        # P(FileType | benign/attack).
        self.file_likelihood = file_likelihood or {
            "video":     {"N": 0.40, "A": 0.10},
            "image":     {"N": 0.30, "A": 0.10},
            "telemetry": {"N": 0.15, "A": 0.05},
            "text":      {"N": 0.07, "A": 0.30},
            "zip":       {"N": 0.08, "A": 0.45},   # worked example type
            "archive":   {"N": 0.08, "A": 0.45},   # archives behave like zip
        }

    # ------- unknown-label fallbacks (kept pessimistic for safety) -------
    _UNKNOWN_ZONE = {"N": 0.10, "A": 0.20}
    _UNKNOWN_FILE = {"N": 0.15, "A": 0.25}

    def _get_zone(self, zone: str) -> Dict[str, float]:
        return self.zone_likelihood.get(zone.lower(), self._UNKNOWN_ZONE)

    def _get_file(self, file_type: str) -> Dict[str, float]:
        return self.file_likelihood.get(file_type.lower(), self._UNKNOWN_FILE)

    def compute_initial_reputation(self, zone: str, file_type: str) -> Dict[str, Any]:
        z = self._get_zone(zone)
        f = self._get_file(file_type)

        num = self.P_N * z["N"] * f["N"]
        den = num + (self.P_A * z["A"] * f["A"])
        R = (num / den) if den > 0 else self.P_N

        return {
            "R": round(R, 6),
            "P_N": self.P_N, "P_A": self.P_A,
            "P_Z_given_N": z["N"], "P_Z_given_A": z["A"],
            "P_F_given_N": f["N"], "P_F_given_A": f["A"],
            "numerator": round(num, 6),
            "denominator": round(den, 6),
            "zone": zone, "file_type": file_type,
        }


# ---- Reproduce the 8 April plan's worked example (expect R ~= 0.257) ----
est = BayesianReputationEstimator(prior_benign=0.85)
out = est.compute_initial_reputation(zone="zone-3", file_type="zip")
print("Plan worked example — expected R ~= 0.257")
for k, v in out.items():
    print(f"  {k:>17}: {v}")
assert abs(out["R"] - 0.257) < 1e-3, f"posterior R={out['R']} does not match plan"
print("\n[OK] Bayesian posterior matches plan's reference value.")


## 7.6 Reputation Update — Reward & Penalty

Once the Multi-Layer Malware Detection Engine returns a verdict, we update $R$ using an **asymmetric** rule so that a single confirmed infection costs much more than a single clean delivery gains:

$$
R_{\text{new}} \;=\;
\begin{cases}
\text{clip}\!\big(R_{\text{old}} + \eta_{+}(1 - R_{\text{old}}),\,0,\,1\big) & \text{if verdict = benign (reward)} \\[4pt]
\text{clip}\!\big(R_{\text{old}} - \eta_{-}\,R_{\text{old}},\,0,\,1\big) & \text{if verdict = malicious (penalty)}
\end{cases}
$$

Defaults: $\eta_{+} = 0.05$ (small reward), $\eta_{-} = 0.50$ (large penalty).

Intuition: the reward pushes $R$ toward 1 proportionally to the room left ($1 - R_{\text{old}}$), while the penalty strips away half the current trust on each confirmed malicious event.


In [ ]:
# 7.7 Reputation update — reward / penalty implementation

def update_reputation(R: float, verdict: str,
                      reward_rate: float = 0.05,
                      penalty_rate: float = 0.50) -> float:
    """Asymmetric Bernoulli-style update driven by detection-engine verdicts."""
    if verdict == "benign":
        R_new = R + reward_rate * (1.0 - R)
    elif verdict == "malicious":
        R_new = R - penalty_rate * R
    else:
        R_new = R   # "unknown"/"quarantined" verdicts leave R unchanged
    return max(0.0, min(1.0, R_new))


# Start from the Bayesian prior and show both paths
R0 = out["R"]
print(f"R0 (Bayesian prior)          = {R0:.4f}")
print(f"R after 1 benign verdict     = {update_reputation(R0, 'benign'):.4f}")
print(f"R after 1 malicious verdict  = {update_reputation(R0, 'malicious'):.4f}")

# Long-run reward trajectory on a clean drone (10 benign verdicts in a row)
R = R0
for i in range(1, 11):
    R = update_reputation(R, "benign")
print(f"\nR after 10 benign verdicts   = {R:.4f}   (should trend toward 1.0)")


# 8. Adaptive $T_S$ Thresholds

## 8.1 Why Static Thresholds Fail

In v2 the inspection routing used hard-coded cut-offs:

| Band | Default | Maps to |
| :--- | :---: | :--- |
| $T_S < 0.4$ | $\theta_{\text{low}}$ | Signature only |
| $0.4 \le T_S < 0.7$ | (band) | Signature + ML |
| $T_S \ge 0.7$ | $\theta_{\text{high}}$ | Sandbox |

⚠️ **Static thresholds fail under evolving attack patterns.** As adversaries adapt, fixed $0.4 / 0.7$ will either flood the sandbox with false positives or miss stealthier attacks.

**Solution:** the Security Feedback Loop adapts $\theta_{\text{low}}$ and $\theta_{\text{high}}$ **dynamically** as live detection metrics drift.

## 8.2 System Performance Monitoring

Signals surfaced by the Response & Quarantine Manager and Security Dashboard:

| Category | Signals |
| :--- | :--- |
| **Detection Metrics** | False Positive Rate (FPR), False Negative Rate (FNR), Precision, Recall |
| **Resource Metrics** | Sandbox Utilization, Latency, Throughput |
| **Target Objectives (example)** | $\text{FPR} < 5\%$, $\text{FNR} < 3\%$, Sandbox Load $< 70\%$ |

## 8.3 Soft Threshold Update Rule

Use a small learning rate $\eta$ for gradual updates:

$$
\theta_{\text{new}} \;=\; \theta_{\text{old}} + \eta \cdot (\text{Performance Error})
$$

For $\theta_{\text{high}}$ (the Medium $\to$ High boundary), balance the two error signals:

$$
\theta_{\text{high, new}} \;=\; \theta_{\text{high, old}} \;+\; \eta\,(\text{FPR} - \text{FPR}_{\text{target}}) \;-\; \eta\,(\text{FNR} - \text{FNR}_{\text{target}})
$$

**Intuition:**

- **High FPR** → increase $\theta_{\text{high}}$ → fewer files hit the sandbox → reduce false alarms.
- **High FNR** → decrease $\theta_{\text{high}}$ → more files hit the sandbox → improve detection.

$\theta_{\text{low}}$ is updated symmetrically for the Low $\to$ Medium boundary.

## 8.4 Final Adaptive Workflow

1. **Monitor detection outcomes** (feedback from the Response & Quarantine Manager).
2. **Compute deviation** from target FPR / FNR.
3. **Apply soft update** using learning rate $\eta$.
4. **Clamp thresholds** within safe bounds (e.g. $\theta_{\text{low}} \in [0.20, 0.50]$, $\theta_{\text{high}} \in [0.55, 0.85]$, and $\theta_{\text{high}} - \theta_{\text{low}} \ge \text{min\_gap}$).

**Why soft updates?**

- Prevents sandbox overload during error spikes.
- Ensures stability — no oscillatory behaviour.
- Gradual convergence to the operating point that satisfies live targets.


In [ ]:
# 8.5 AdaptiveThresholdManager — runnable implementation
from dataclasses import dataclass, field
from typing import List, Tuple


@dataclass
class AdaptiveThresholdManager:
    th_low: float = 0.40
    th_high: float = 0.70

    # Target detection objectives (example values from the plan).
    FPR_target: float = 0.05
    FNR_target: float = 0.03

    # Learning rate for soft updates (small -> stable, avoids oscillation).
    eta: float = 0.10

    # Safe bounds to prevent pathological thresholds.
    low_min: float = 0.20
    low_max: float = 0.50
    high_min: float = 0.55
    high_max: float = 0.85
    min_gap: float = 0.10   # th_high - th_low must stay >= min_gap

    history: List[dict] = field(default_factory=list)

    # ------------- helpers -------------
    def _clamp(self) -> None:
        self.th_low  = max(self.low_min,  min(self.low_max,  self.th_low))
        self.th_high = max(self.high_min, min(self.high_max, self.th_high))
        if (self.th_high - self.th_low) < self.min_gap:
            self.th_high = min(self.high_max, self.th_low + self.min_gap)

    # ------------- public API -------------
    def update(self, fpr: float, fnr: float) -> Tuple[float, float]:
        """Soft-update both thresholds from observed FPR / FNR."""
        fpr_err = fpr - self.FPR_target
        fnr_err = fnr - self.FNR_target

        # theta_high controls Medium -> High (sandbox) routing.
        #   High FPR -> raise theta_high (fewer sandboxes).
        #   High FNR -> lower theta_high (more sandboxes).
        self.th_high = self.th_high + self.eta * fpr_err - self.eta * fnr_err

        # theta_low controls Low -> Medium (ML) routing; same logic,
        # smaller effective reach, so share the update rule.
        self.th_low  = self.th_low  + self.eta * fpr_err - self.eta * fnr_err

        self._clamp()
        self.history.append({
            "fpr": fpr, "fnr": fnr,
            "th_low": self.th_low, "th_high": self.th_high,
        })
        return self.th_low, self.th_high

    def classify(self, T_S: float) -> str:
        """Map a threat score to an inspection level using *live* thresholds."""
        if T_S < self.th_low:
            return "Low"
        if T_S < self.th_high:
            return "Medium"
        return "High"


# ---- Demonstration: drift the thresholds over a few monitoring cycles ----
mgr = AdaptiveThresholdManager()
print(f"{'phase':25s} th_low   th_high")
print(f"{'start (defaults)':25s} {mgr.th_low:.3f}    {mgr.th_high:.3f}")

mgr.update(fpr=0.12, fnr=0.02)   # sandbox flooded by false positives
print(f"{'after high FPR (.12/.02)':25s} {mgr.th_low:.3f}    {mgr.th_high:.3f}")

mgr.update(fpr=0.03, fnr=0.08)   # missing real attacks
print(f"{'after high FNR (.03/.08)':25s} {mgr.th_low:.3f}    {mgr.th_high:.3f}")

mgr.update(fpr=0.05, fnr=0.03)   # metrics back on target
print(f"{'converged (.05/.03)':25s} {mgr.th_low:.3f}    {mgr.th_high:.3f}")


# 9. Integrated v3 Pipeline — Bayesian $R$ + Game-Theoretic $T_S$ + Adaptive Thresholds

This section stitches Steps 1 – 5 (the v2 game-theoretic Stackelberg estimator) together with §7 (Bayesian initial $R$) and §8 (adaptive thresholds):

1. **Ingestion** yields `mission_zone` and the dominant artifact type.
2. **Reputation router:**
    - If the drone has history in `reputation_db`, use the stored $R$.
    - Else, compute the **Bayesian initial $R$** via `BayesianReputationEstimator`.
3. $R$ flows into **Step 2** of v2 to produce $I'$ and $\text{DSR}'$.
4. **Steps 3 – 4** produce the Stackelberg equilibrium and the raw $T_S$.
5. The `AdaptiveThresholdManager` maps $T_S$ to the inspection level using **live** $\theta_{\text{low}} / \theta_{\text{high}}$.
6. After the detection engine returns a verdict, feed it back:
    - `update_reputation(R, verdict)` adjusts the stored $R$.
    - `AdaptiveThresholdManager.update(FPR, FNR)` adjusts the thresholds.

The only change to the v2 core is that $R$ is now sourced intelligently and the threshold constants are replaced with a live manager — all existing formulas (impact, DSR, payoffs, equilibrium, $T_S$) stay untouched.


In [ ]:
# 9.1 End-to-end v3 pipeline — uses Step-5 helpers already defined above.

def dominant_file_type(artifact_records):
    """Pick the highest-risk type present; falls back to the first type."""
    if not artifact_records:
        return "other"
    priority = ["zip", "archive", "video", "image", "telemetry", "text"]
    present = {a.get("type", "other").lower() for a in artifact_records}
    for t in priority:
        if t in present:
            return t
    return next(iter(present))


def v3_estimate(ingestion, reputation_db=None, bayes=None, thr_mgr=None,
                Z=0.5, H=0.0, TI=0.0):
    """Run the v3 estimator on a single ingestion record.

    Parameters
    ----------
    reputation_db : dict mapping drone_id -> R (historical reputation store).
    bayes         : BayesianReputationEstimator used when drone has no history.
    thr_mgr       : AdaptiveThresholdManager that owns live th_low / th_high.
    Z, H, TI      : zone-risk, history freq, threat-intel strength (as in v2).
    """
    reputation_db = reputation_db if reputation_db is not None else {}
    bayes   = bayes   or BayesianReputationEstimator()
    thr_mgr = thr_mgr or AdaptiveThresholdManager()

    md = ingestion["ingest_metadata"]
    ar = ingestion["artifact_records"]
    drone_id  = md["drone_id"]
    zone      = md.get("mission_zone", "unknown")
    file_type = dominant_file_type(ar)

    # ---- (1) Reputation: history if we have it, else Bayesian prior. ----
    if drone_id in reputation_db:
        R = reputation_db[drone_id]
        R_source = "history"
        bayes_trace = None
    else:
        bayes_trace = bayes.compute_initial_reputation(zone, file_type)
        R = bayes_trace["R"]
        R_source = "bayesian_prior"

    # ---- (2) Game-theoretic estimator (unchanged v2 code). ----
    I_base    = compute_I_base(ar, md)
    I_prime   = compute_I_prime(I_base, R, Z)
    DSR_prime = compute_DSR_primes(DSR_base, H, TI)
    U_a_mat, U_d_mat = build_payoff_matrices(I_prime, DSR_prime)
    eq      = solve_stackelberg_pure(U_a_mat, U_d_mat)
    scored  = compute_threat_score(eq["U_a_eq"], eq["U_d_eq"], R)

    # ---- (3) Inspection level via ADAPTIVE thresholds (not static 0.4/0.7). ----
    level = thr_mgr.classify(scored["T_S"])

    return {
        "drone_id": drone_id, "zone": zone, "file_type": file_type,
        "R": round(R, 6), "R_source": R_source,
        "I_base": I_base, "I_prime": round(I_prime, 6),
        "U_d_eq": eq["U_d_eq"], "U_a_eq": eq["U_a_eq"],
        "T_S": scored["T_S"],
        "inspection_level": level,
        "thresholds": {"th_low": thr_mgr.th_low, "th_high": thr_mgr.th_high},
        "bayes_trace": bayes_trace,
    }


# ---- Run v3 pipeline on the four sample ingestions (all cold-start). ----
bayes_est = BayesianReputationEstimator()
thr_mgr   = AdaptiveThresholdManager()
rep_db    = {}   # empty -> every drone is treated as cold-start

print(f"{'drone':8s} {'zone':6s} {'type':10s} {'R(prior)':>9s} {'source':>16s} {'T_S':>7s}  inspection")
print("-" * 76)
for sample in [sampleA, sampleB, sampleC, sampleD, sampleHighRisk]:
    res = v3_estimate(sample, reputation_db=rep_db, bayes=bayes_est, thr_mgr=thr_mgr)
    print(f"{res['drone_id']:8s} {res['zone']:6s} {res['file_type']:10s} "
          f"{res['R']:>9.3f} {res['R_source']:>16s} {res['T_S']:>7.3f}  {res['inspection_level']}")


## 9.2 Feedback-Loop Demonstration

Let the adaptive manager see a few monitoring windows and observe both the threshold drift and how the *same* $T_S$ can change inspection level as thresholds move.


In [ ]:
# 9.2 Feedback-loop demo — threshold drift and inspection-level flip.
import random

random.seed(0)
bayes_est = BayesianReputationEstimator()
thr_mgr   = AdaptiveThresholdManager()

# Pretend the system observes 7 monitoring windows where FPR is initially
# too high and gradually settles, while FNR hovers near target.
print(f"{'window':>6s} {'FPR':>6s} {'FNR':>6s} -> {'th_low':>7s} {'th_high':>8s}")
for i in range(1, 8):
    fpr = max(0.02, 0.12 - 0.01 * i + random.uniform(-0.01, 0.01))
    fnr = max(0.01, 0.04 + random.uniform(-0.01, 0.01))
    thr_mgr.update(fpr, fnr)
    print(f"{i:>6d} {fpr:>6.3f} {fnr:>6.3f} -> {thr_mgr.th_low:>7.3f} {thr_mgr.th_high:>8.3f}")

# Same T_S, different inspection decisions depending on where thresholds are now.
T_S_probe = 0.48
print(f"\nProbe T_S = {T_S_probe}")
print(f"  with current thresholds ({thr_mgr.th_low:.3f} / {thr_mgr.th_high:.3f}) "
      f"=> {thr_mgr.classify(T_S_probe)}")

thr_static_low, thr_static_high = 0.40, 0.70
static_level = ("Low" if T_S_probe < thr_static_low
                else "Medium" if T_S_probe < thr_static_high
                else "High")
print(f"  with static v2 thresholds ({thr_static_low} / {thr_static_high})       "
      f"=> {static_level}")


## 9.3 End-to-End Loop: Cold-Start → Verdict → Reputation & Threshold Update

Finally, close the loop: start from an empty reputation DB, run the estimator, simulate a detection-engine verdict, and let both the reputation and the adaptive thresholds absorb the feedback.


In [ ]:
# 9.3 Close the feedback loop end-to-end.

bayes_est = BayesianReputationEstimator()
thr_mgr   = AdaptiveThresholdManager()
rep_db    = {}

# 1) Cold-start evaluation for DRN-002 (encrypted ZIP, suspicious-looking).
res1 = v3_estimate(sampleB, reputation_db=rep_db, bayes=bayes_est, thr_mgr=thr_mgr)
print("--- cold-start evaluation ---")
print(f"  R(prior) = {res1['R']:.3f}   (source: {res1['R_source']})")
print(f"  T_S      = {res1['T_S']:.3f}   inspection: {res1['inspection_level']}")

# 2) Detection engine returns a verdict (simulate 'malicious').
verdict = "malicious"
rep_db[res1["drone_id"]] = update_reputation(res1["R"], verdict)
print(f"\n--- after '{verdict}' verdict ---")
print(f"  stored R = {rep_db[res1['drone_id']]:.3f}")

# Aggregate metrics from this monitoring window (pretend FPR too high, FNR low).
thr_mgr.update(fpr=0.09, fnr=0.02)
print(f"  th_low / th_high = {thr_mgr.th_low:.3f} / {thr_mgr.th_high:.3f}")

# 3) Re-evaluate the same drone — this time reputation comes from history.
res2 = v3_estimate(sampleB, reputation_db=rep_db, bayes=bayes_est, thr_mgr=thr_mgr)
print("\n--- re-evaluation with updated state ---")
print(f"  R        = {res2['R']:.3f}   (source: {res2['R_source']})")
print(f"  T_S      = {res2['T_S']:.3f}   inspection: {res2['inspection_level']}")
print(f"  thresholds in use = {res2['thresholds']}")
